# RAG Grounding

## Objective
Attach explicit metadata and review evidence to retrieved candidate items so later stages can explain *why* an item is on the table, not just that it was retrieved. This notebook builds on the structured request and retrieval setup from earlier stages, but keeps grounding narrow: it supports a small candidate set with evidence rather than acting as a second recommender.

## Inputs
- `../data/items.parquet`
- `../data/reviews.parquet`
- retrieval artifacts produced by notebook 02
- curated scenarios from `../data/curated/scenario_suite.json`
- shared helpers in `functions/`

## Outputs
- evidence rows for retrieved candidates
- worked examples showing when metadata is enough and when reviews add value
- a grounding-ready evidence structure that notebook 06 can reuse


## What Grounding Is Responsible For

Grounding stays narrow by design. It does **not** rank the full catalog. It operates only on a small candidate set and asks a different question: if we present this item to a user, what metadata or review text could reasonably support that decision?


In [7]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / "functions").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root containing functions/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from functions.grounding import assemble_grounding_evidence
from functions.io import (
    connect_catalog_views,
    load_cross_encoder,
    load_curated_scenarios,
    load_retrieval_artifacts,
    materialize_scenarios,
)
from functions.parsing import build_parser_catalog, parse_user_request
from functions.retrieval import retrieve_candidates_for_request

artifacts = load_retrieval_artifacts(load_encoder_model=True)
candidate_pool = artifacts["candidate_pool"]
index_metadata = artifacts["index_metadata"]
parser_catalog = build_parser_catalog(candidate_pool)
scenario_suite = load_curated_scenarios()
demo_scenarios = materialize_scenarios(scenario_suite["demo_scenarios"], candidate_pool)
con = connect_catalog_views(include_reviews=True)
cross_encoder = load_cross_encoder()

baseline_summary = pd.DataFrame(
    {
        "metric": ["baseline_name", "candidate_rows", "price_policy"],
        "value": [
            index_metadata.get("baseline_name"),
            len(candidate_pool),
            index_metadata.get("candidate_pool_strategy", {}).get("price_policy"),
        ],
    }
)

display(baseline_summary)
display(candidate_pool.head(3))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,metric,value
0,baseline_name,metadata_hybrid_v3_300k_source_aware
1,candidate_rows,300000
2,price_policy,Exclude items with null price from the retrievable pool because they were likely not purchasable when the snapshot was collected.


,retrieval_row_id,parent_asin,source,main_category,title,store,average_rating,rating_number,price,retrieval_text,retrieval_text_chars
0,0,B07KTYJ769,electronics,Amazon Devices,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device",Amazon,4.7,542825,24.99,"Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device Amazon Smart Plug, for home automation, Works with Alexa - A Certified for Humans Device Amazon Amazon Devi...",200
1,1,B0BGNG1294,electronics,Home Audio & Theater,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black",Amazon Basics,4.7,507202,8.55,"Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black Amazon Basics HDMI Cable, 18Gbps High-Speed, 4K@60Hz, 2160p, Ethernet Ready, 10 Foot, Black Amazon Basic...",288
2,2,B08CLNX58K,electronics,Computers,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT,SanDisk,4.7,402800,13.40,SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT SanDisk 32GB 2-Pack Ultra MicroSDHC UHS-I Memory Card (2x32GB) - SDSQUAR-032G-GN6MT SanDisk Computers Electronic...,292


## Evidence Objects

The evidence unit stays simple enough to inspect. Each row is either a metadata block or a review snippet, and later notebooks should be able to tell where it came from, which claim it seems to support, and how strongly it matched the request.


In [8]:
evidence_object_schema = pd.DataFrame(
    [
        {"field": "parent_asin", "purpose": "links evidence back to the candidate item"},
        {"field": "candidate_title", "purpose": "keeps evidence rows readable during inspection"},
        {"field": "evidence_source", "purpose": "distinguishes metadata from review evidence"},
        {"field": "evidence_id", "purpose": "stable identifier for later joins or exports"},
        {"field": "evidence_text", "purpose": "the text block that could support an explanation"},
        {"field": "relevance_score", "purpose": "cross-encoder relevance to the grounding query"},
        {"field": "matched_terms", "purpose": "which request terms visibly overlap with the evidence"},
        {"field": "matched_topic", "purpose": "coarse explanation topic inferred from the evidence row"},
        {"field": "grounding_need", "purpose": "preserves the claim category requested upstream"},
        {
            "field": "review_title / helpful_vote / verified_purchase",
            "purpose": "review-specific supporting metadata when available",
        },
    ]
)

display(evidence_object_schema)

,field,purpose
0,parent_asin,links evidence back to the candidate item
1,candidate_title,keeps evidence rows readable during inspection
2,evidence_source,distinguishes metadata from review evidence
3,evidence_id,stable identifier for later joins or exports
4,evidence_text,the text block that could support an explanation
5,relevance_score,cross-encoder relevance to the grounding query
6,matched_terms,which request terms visibly overlap with the evidence
7,matched_topic,coarse explanation topic inferred from the evidence row
8,grounding_need,preserves the claim category requested upstream
9,review_title / helpful_vote / verified_purchase,review-specific supporting metadata when available


## Worked Examples

The examples below cover different grounding situations: one where structured metadata already says a lot, one where a reference item matters, one where review evidence adds nuance, and one where the request remains partly ambiguous even after retrieval.


In [9]:
example_labels = [
    "budget_headphones_avoid_beats",
    "reference_cheaper_headphones",
    "lightweight_programming_laptop",
    "gift_ambiguity",
]
selected_scenarios = [scenario for scenario in demo_scenarios if scenario["label"] in example_labels]

worked_examples = []
for scenario in selected_scenarios:
    parsed = parse_user_request(
        scenario["query"],
        parser_catalog,
        reference_item_context=scenario["reference_item_context"],
    )
    candidates = retrieve_candidates_for_request(parsed, artifacts, top_k=3)
    evidence = assemble_grounding_evidence(
        parsed,
        candidates,
        con,
        candidate_pool,
        cross_encoder,
        top_review_evidence_per_item=1,
    )
    worked_examples.append(
        {
            "label": scenario["label"],
            "title": scenario["title"],
            "why_it_is_interesting": scenario["why_it_is_interesting"],
            "parsed_request": parsed,
            "candidates": candidates,
            "evidence": evidence,
        }
    )

example_summary = pd.DataFrame(
    [
        {
            "label": example["label"],
            "candidate_count": len(example["candidates"]),
            "metadata_rows": int((example["evidence"]["evidence_source"] == "metadata").sum()),
            "review_rows": int((example["evidence"]["evidence_source"] == "review").sum()),
            "clarification_needed": example["parsed_request"]["clarification_needed"],
        }
        for example in worked_examples
    ]
)

display(example_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,label,candidate_count,metadata_rows,review_rows,clarification_needed
0,budget_headphones_avoid_beats,3,3,3,False
1,reference_cheaper_headphones,3,3,3,False
2,lightweight_programming_laptop,3,3,3,False
3,gift_ambiguity,3,3,3,True


In [10]:
for example in worked_examples:
    print("=" * 100)
    print(example["label"])
    print(example["why_it_is_interesting"])
    print("Parsed request")
    print(json.dumps(example["parsed_request"], indent=2, ensure_ascii=False))

    print("Candidates")
    display(
        example["candidates"][
            ["parent_asin", "source", "title", "store", "price", "average_rating", "retrieval_score"]
        ]
    )

    print("Grounding evidence")
    display(
        example["evidence"][
            [
                "parent_asin",
                "candidate_title",
                "evidence_source",
                "relevance_score",
                "matched_topic",
                "matched_terms",
                "review_title",
                "helpful_vote",
                "verified_purchase",
                "evidence_text",
            ]
        ]
    )

budget_headphones_avoid_beats
Combines product type, budget sensitivity, quality language, and a brand exclusion in one request.
Parsed request
{
  "original_query": "Recommend wireless bluetooth headphones under 80 dollars that are highly rated, but avoid Beats.",
  "user_intent": "constrained_search",
  "domain_hint": "electronics",
  "hard_constraints": {
    "max_price": 80.0,
    "min_rating": null,
    "must_include_terms": [
      "headphones"
    ],
    "use_case": null,
    "cheaper_than_reference": false,
    "same_source_as_reference": false
  },
  "soft_preferences": [
    "highly_rated"
  ],
  "reference_items": [],
  "excluded_brands": [
    "Beats"
  ],
  "excluded_terms": [],
  "grounding_needs": [
    "price",
    "rating",
    "brand_constraint"
  ],
  "clarification_needed": false,
  "clarification_questions": []
}
Candidates


,parent_asin,source,title,store,price,average_rating,retrieval_score
0,B081RBT9HM,electronics,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel",Kids Disney Headphones,19.99,4.3,0.122747
1,B00C4FBQT8,electronics,Sony 900MHz Wireless Stereo Noise Reduction Headphones,Sony,119.95,3.9,0.106422
2,B0BZ6QTH4S,electronics,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple",Philips,19.99,4.4,0.106393


Grounding evidence


,parent_asin,candidate_title,evidence_source,relevance_score,matched_topic,matched_terms,review_title,helpful_vote,verified_purchase,evidence_text
0,B081RBT9HM,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel",metadata,-2.293530,price,"[headphones, price, rating]",None,NaN,None,"Title: Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel Description..."
1,B081RBT9HM,"Black Panther Headphones, Adjustable Headband, Stereo Sound, 3.5Mm Jack, Wired Headphones, Tangle-Free, Volume Control, Foldable, Kids Headphones Over Ear for School Home Travel",review,-0.551716,rating,"[review, headphones, are]",DO NOT BUY!!!,0.0,True,"Review title: DO NOT BUY!!!. Review: I wish I could give it a no stars because these headphones are trash!! You could barely hear anything, quality wasn’t good. Very disappointed"
2,B00C4FBQT8,Sony 900MHz Wireless Stereo Noise Reduction Headphones,metadata,-0.058744,price,"[wireless, headphones, price, rating]",None,NaN,None,"Title: Sony 900MHz Wireless Stereo Noise Reduction Headphones Description: Sony 900MHz RF wireless transmission Wireless Stereo Noise Reduction Headphone System , These wireless headphones from So..."
3,B00C4FBQT8,Sony 900MHz Wireless Stereo Noise Reduction Headphones,review,2.103269,rating,"[review, headphones, but, wireless, under, are]","Imperfect compared to wired headphones, but perhaps as good as wireless headphones get under $100",6.0,True,"Review title: Imperfect compared to wired headphones, but perhaps as good as wireless headphones get under $100. Review: I've tried a few RF wireless headphones for TV use now, and these are the b..."
4,B0BZ6QTH4S,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple",metadata,-3.062499,price,"[headphones, price, rating]",None,NaN,None,"Title: PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple Description: Sized for kids, maximum volume limited. The right headphones to introduce Mini..."
5,B0BZ6QTH4S,"PHILIPS Coolplay Kids On-Ear Headphones - 85dB Volume Limiter - Safer Hearing (SHK2000PK), Purple",review,-1.259162,rating,"[review, headphones]",Perfect for grade school age,1.0,True,Review title: Perfect for grade school age. Review: Love these headphones for my son! Safe on the ears. not too loud!


reference_cheaper_headphones
Tests context-aware parsing and whether the pipeline can stay anchored on a real reference item while rewarding lower price.
Parsed request
{
  "original_query": "I want something like this item but cheaper.",
  "user_intent": "similar_item_refinement",
  "domain_hint": "electronics",
  "hard_constraints": {
    "max_price": 21.99,
    "min_rating": null,
    "must_include_terms": [],
    "use_case": null,
    "cheaper_than_reference": true,
    "same_source_as_reference": true
  },
  "soft_preferences": [
    "budget_friendly"
  ],
  "reference_items": [
    {
      "mention": "this item",
      "resolved_parent_asin": "B08XPWDSWW",
      "resolved_title": "TOZO T10 Bluetooth 5.3 Wireless Earbuds with Wireless Charging Case IPX8 Waterproof Stereo Headphones in Ear Built in Mic Headset Premium Sound with Deep Bass for Sport White (2022 Upgraded)",
      "source": "electronics",
      "store": "TOZO",
      "price": 21.99,
      "resolution_status": "from_co

...

,parent_asin,candidate_title,evidence_source,relevance_score,matched_topic,matched_terms,review_title,helpful_vote,verified_purchase,evidence_text
0,B01N5E5T48,"ASUS L402SA Portable Lightweight Laptop PC, Intel Dual Core Processor, 4GB RAM, 32GB Flash Storage with Windows 10 with 1 Year Microsoft Office 365 Subscription",metadata,0.134369,use_case_fit,"[portable, lightweight, laptop, compact, for]",None,NaN,None,"Title: ASUS L402SA Portable Lightweight Laptop PC, Intel Dual Core Processor, 4GB RAM, 32GB Flash Storage with Windows 10 with 1 Year Microsoft Office 365 Subscription Description: ASUS L402SA Por..."
1,B01N5E5T48,"ASUS L402SA Portable Lightweight Laptop PC, Intel Dual Core Processor, 4GB RAM, 32GB Flash Storage with Windows 10 with 1 Year Microsoft Office 365 Subscription",review,-5.609837,use_case_fit,"[a, laptop, for, portable]",This is not a bad laptop. I bought this for my fiance as ...,3.0,True,Review title: This is not a bad laptop. I bought this for my fiance as .... Review: This is not a bad laptop. I bought this for my fiance as a portable work laptop for text documents and surfing t...
2,B07YHJ9P5N,"Asus Vivobook E203MA Thin and Lightweight 11.6” HD Laptop, Intel Celeron N4000 Processor, 4GB RAM, 64GB eMMC Storage, 802.11AC Wi-Fi, HDMI, USB-C, Win 10",metadata,-4.641430,use_case_fit,"[lightweight, laptop, for, compact]",None,NaN,None,"Title: Asus Vivobook E203MA Thin and Lightweight 11.6” HD Laptop, Intel Celeron N4000 Processor, 4GB RAM, 64GB eMMC Storage, 802.11AC Wi-Fi, HDMI, USB-C, Win 10 Description: 11.6\ Display> typical..."
3,B07YHJ9P5N,"Asus Vivobook E203MA Thin and Lightweight 11.6” HD Laptop, Intel Celeron N4000 Processor, 4GB RAM, 64GB eMMC Storage, 802.11AC Wi-Fi, HDMI, USB-C, Win 10",review,-5.764547,use_case_fit,"[laptop, for]",Damn good computer,32.0,True,Review title: Damn good computer. Review: Amazing laptop for the price! Nice keyboard as well. Very happy
4,B09NBG2K3G,"Broage 15.6"" FHD Lightweight Laptop Computer, Intel Celeron N4020 up to 2.8GHz, 4GB RAM, 64GB eMMC, WiFi, Bluetooth, USB 3.0, HDMI, Webcam, Microphone, Silver, Windows 10 Home, Online Class Ready",metadata,-2.452380,use_case_fit,"[lightweight, laptop, a, for]",None,NaN,None,"Title: Broage 15.6"" FHD Lightweight Laptop Computer, Intel Celeron N4020 up to 2.8GHz, 4GB RAM, 64GB eMMC, WiFi, Bluetooth, USB 3.0, HDMI, Webcam, Microphone, Silver, Windows 10 Home, Online Class..."
5,B09NBG2K3G,"Broage 15.6"" FHD Lightweight Laptop Computer, Intel Celeron N4020 up to 2.8GHz, 4GB RAM, 64GB eMMC, WiFi, Bluetooth, USB 3.0, HDMI, Webcam, Microphone, Silver, Windows 10 Home, Online Class Ready",review,-1.973168,use_case_fit,"[lightweight, laptop, use]",Great lightweight laptop of excellent quality.,2.0,True,Review title: Great lightweight laptop of excellent quality.. Review: I like this product because it was affordable while not being of cheap quality. This laptop is easy to use and has all of the ...


gift_ambiguity
This case should remain conservative and preserve ambiguity rather than pretending the product category is obvious.
Parsed request
{
  "original_query": "Find me something good for gifting, not too expensive.",
  "user_intent": "gift_search",
  "domain_hint": null,
  "hard_constraints": {
    "max_price": null,
    "min_rating": null,
    "must_include_terms": [],
    "use_case": "gifting",
    "cheaper_than_reference": false,
    "same_source_as_reference": false
  },
  "soft_preferences": [
    "budget_friendly",
    "giftable"
  ],
  "reference_items": [],
  "excluded_brands": [],
  "excluded_terms": [],
  "grounding_needs": [
    "use_case_fit",
    "giftability"
  ],
  "clarification_needed": true,
  "clarification_questions": [
    "What kind of gift or product category do you have in mind?"
  ]
}
Candidates


,parent_asin,source,title,store,price,average_rating,retrieval_score
0,B0C5S9WGT7,home_and_kitchen,SereneLife Bamboo Bathtub Caddy with Luxury Gift Box and Red Gifting Ribbon Extendable & Adjustable Tray with Device/Book Holder with Removable Trays for Bath Accessories (Stain Brown),SereneLife,29.31,4.7,0.196393
1,B09LFLPSHM,home_and_kitchen,"Paksh Novelty Italian White Wine Glasses - - Wine Glass Set for Parties, Weddings, Gifting - Clear Wine Glass, for Red and White Wine - Christmas Gift for Women & Men",Paksh Novelty,70.40,4.7,0.196129
2,B01NCQ5EPW,home_and_kitchen,"Oud Bakhoor Variety Box دخني عود بخور by Dukhni | Assorted Box | 30 Pieces Bakhoor | Gift Set & Refill Kit | Arabic Bakhoor Incense | Perfect for Ramadan Prayer and Gifting | Luxurious, Long Lasting",Dukhni,19.99,4.5,0.195873


Grounding evidence


,parent_asin,candidate_title,evidence_source,relevance_score,matched_topic,matched_terms,review_title,helpful_vote,verified_purchase,evidence_text
0,B0C5S9WGT7,SereneLife Bamboo Bathtub Caddy with Luxury Gift Box and Red Gifting Ribbon Extendable & Adjustable Tray with Device/Book Holder with Removable Trays for Bath Accessories (Stain Brown),metadata,-8.722461,use_case_fit,"[gift, gifting, for, use, price]",None,NaN,None,Title: SereneLife Bamboo Bathtub Caddy with Luxury Gift Box and Red Gifting Ribbon Extendable & Adjustable Tray with Device/Book Holder with Removable Trays for Bath Accessories (Stain Brown) Desc...
1,B0C5S9WGT7,SereneLife Bamboo Bathtub Caddy with Luxury Gift Box and Red Gifting Ribbon Extendable & Adjustable Tray with Device/Book Holder with Removable Trays for Bath Accessories (Stain Brown),review,-2.409353,use_case_fit,[affordable],Exactly what I wanted!,0.0,True,"Review title: Exactly what I wanted!. Review: Totally worth every penny, plus it was very affordable. Perfect!"
2,B09LFLPSHM,"Paksh Novelty Italian White Wine Glasses - - Wine Glass Set for Parties, Weddings, Gifting - Clear Wine Glass, for Red and White Wine - Christmas Gift for Women & Men",metadata,-7.228302,use_case_fit,"[for, gifting, gift, price]",None,NaN,None,"Title: Paksh Novelty Italian White Wine Glasses - - Wine Glass Set for Parties, Weddings, Gifting - Clear Wine Glass, for Red and White Wine - Christmas Gift for Women & Men Features: LUXURIOUS DE..."
3,B09LFLPSHM,"Paksh Novelty Italian White Wine Glasses - - Wine Glass Set for Parties, Weddings, Gifting - Clear Wine Glass, for Red and White Wine - Christmas Gift for Women & Men",review,-4.707969,use_case_fit,"[for, expensive]",Five Stars,4.0,True,Review title: Five Stars. Review: Perfect tall glasses for the table. Makes your wine taste expensive. (a mind trick)
4,B01NCQ5EPW,"Oud Bakhoor Variety Box دخني عود بخور by Dukhni | Assorted Box | 30 Pieces Bakhoor | Gift Set & Refill Kit | Arabic Bakhoor Incense | Perfect for Ramadan Prayer and Gifting | Luxurious, Long Lasting",metadata,-7.180706,use_case_fit,"[gift, for, gifting, experience, price]",None,NaN,None,"Title: Oud Bakhoor Variety Box دخني عود بخور by Dukhni | Assorted Box | 30 Pieces Bakhoor | Gift Set & Refill Kit | Arabic Bakhoor Incense | Perfect for Ramadan Prayer and Gifting | Luxurious, Lo..."
5,B01NCQ5EPW,"Oud Bakhoor Variety Box دخني عود بخور by Dukhni | Assorted Box | 30 Pieces Bakhoor | Gift Set & Refill Kit | Arabic Bakhoor Incense | Perfect for Ramadan Prayer and Gifting | Luxurious, Long Lasting",review,-3.350008,use_case_fit,[good],good,2.0,True,Review title: good. Review: I like it ،my favorite is oud ya aini 😍😍


## Limits Of This Grounding Layer

Grounding is conservative. Reviews can be noisy, evidence can be sparse, and some explanation topics remain weak when metadata and reviews do not line up cleanly. That is an acceptable boundary for this stage: future notebooks can build on explicit evidence instead of inventing justifications from scratch.
